In [2]:
import torch
import torch.nn as nn
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f'device : {device}')

device : cpu


LSTM은 한 시점 $t$에서 아래 4가지를 계산합니다.

- 망각 게이트: $f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)$
- 입력 게이트: $i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)$
- 후보 정보: $c̃_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)$
- 출력 게이트: $o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)$

그리고 기억과 출력은 다음처럼 계산합니다.

- $c_t = f_t \cdot c_{t-1} + i_t \cdot c̃_t$
- $h_t = o_t \cdot \tanh(c_t)$

In [ ]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def step_lstm_manual(x_t, h_prev, c_prev, weights):
    W_f, U_f, b_f = weights['W_f'], weights['U_f'], weights['b_f']
    W_i, U_i, b_i = weights['W_i'], weights['U_i'], weights['b_i']
    W_c, U_c, b_c = weights['W_c'], weights['U_c'], weights['b_c']
    W_o, U_o, b_o = weights['W_o'], weights['U_o'], weights['b_o']

    f_t = sigmoid(W_f @ x_t + U_f @ h_prev + b_f)
    i_t = sigmoid(W_i @ x_t + U_i @ h_prev + b_i)
    c_hat_t = torch.tanh(W_c @ x_t + U_c @ h_prev + b_c)
    o_t = sigmoid(W_o @ x_t + U_o @ h_prev + b_o)
    c_t = f_t * c_prev + i_t * c_hat_t
    h_t = o_t * torch.tanh(c_t)

    return {
        'f_t': f_t,
        'i_t': i_t,
        'c_hat_t': c_hat_t,
        'o_t': o_t,
        'c_t': c_t,
        'h_t': h_t,
    }

### 한 시점 계산을 직접 해보기
- 입력벡터는 2차원, 은닉상태도 2차원

In [ ]:
x_t = torch.tensor([[2.5], [1.0]])
h_prev = torch.tensor([[1.0], [0.5]])
c_prev = torch.tensor([[3.0], [2.0]])

weights = {
    'W_f': torch.tensor([[0.3, -0.1], [0.2, 0.4]]),
    'U_f': torch.tensor([[0.4, 0.2], [-0.1, 0.3]]),
    'b_f': torch.tensor([[ -0.2], [0.1]]),
    'W_i': torch.tensor([[0.4, 0.1], [0.3, -0.2]]),
    'U_i': torch.tensor([[0.3, 0.2], [0.1, 0.4]]),
    'b_i': torch.tensor([[0.1], [0.0]]),
    'W_c': torch.tensor([[0.5, -0.2], [0.2, 0.3]]),
    'U_c': torch.tensor([[0.2, 0.1], [0.3, -0.1]]),
    'b_c': torch.tensor([[0.3], [-0.2]]),
    'W_o': torch.tensor([[0.35, 0.05], [0.25, -0.15]]),
    'U_o': torch.tensor([[0.5, 0.1], [0.2, 0.3]]),
    'b_o': torch.tensor([[-0.1], [0.2]]),
}

result = step_lstm_manual(x_t, h_prev, c_prev, weights)
for name, value in result.items():
    print(name)
    print(value)
    print()